# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR⁲ dataset using the `mlcroissant` library. All dataset components are referenced by their `@id` as specified by the Croissant standard.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Install the `mlcroissant` library (if not already installed)
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and explore its contents using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Get metadata as a Python dict
metadata_dict = dataset.metadata.to_dict()

# Display main information
dataset_name = metadata_dict.get('name', 'No title')
dataset_description = metadata_dict.get('description', 'No description')
print(f"{dataset_name}\nDescription: {dataset_description}")

## 2. Data Overview
Review available record sets, fields, and their IDs (`@id`).

We list all Record Sets, their fields, and columns by their `@id`.

The primary entities for working with the FAIR² dataset are:
- **Record Sets** (`cr:RecordSet`): high-level tables/collections.
- **Fields** (`cr:Field`): logical elements of a record (analogs to fields in a database).
- **Columns** (`cr:column`): physical columns in data files.

In [ ]:
# Utility: Helper to pretty-print nested info by @id and name
def print_fields(record_set, pad='  '):
    print(f"  Fields for RecordSet {record_set['@id']}:")
    fields = record_set.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for f in fields:
        fid = f.get('@id', str(f)) if isinstance(f, dict) else str(f)
        fname = f.get('name', '') if isinstance(f, dict) else ''
        print(f"    - {fid} {f'({fname})' if fname else ''}")

# 1. List record sets by @id and name
record_sets = metadata_dict.get('recordSet', [])
if isinstance(record_sets, dict):
    record_sets = [record_sets]  # Singleton correction
print("Available Record Sets in the dataset ('@id': name):")
for rs in record_sets:
    rs_id = rs.get('@id', str(rs))
    rs_name = rs.get('name', '')
    print(f"- {rs_id} {f'({rs_name})' if rs_name else ''}")
    print_fields(rs)
    # Also print columns within each record set
    columns = rs.get('column', [])
    if isinstance(columns, dict):
        columns = [columns]
    if columns:
        print(f"  Columns in this RecordSet:")
        for c in columns:
            cid = c.get('@id', str(c)) if isinstance(c, dict) else str(c)
            cname = c.get('name', '') if isinstance(c, dict) else ''
            print(f"    - {cid} {f'({cname})' if cname else ''}")

# If no record sets were found, alert the user
if not record_sets:
    print("No record sets defined in the metadata. The dataset may contain only a single main file.")

## 3. Data Extraction
Load data from a **specific record set** into a DataFrame for analysis.

> **Note**: Replace `<record_set_id>` with the actual RecordSet `@id` shown in the previous section. If the dataset contains only one RecordSet, use that.

In [ ]:
# Automatically use all listed record sets, or default to all available if missing
if record_sets:
    record_set_ids = [rs['@id'] for rs in record_sets]
else:
    # Use main distribution file if no explicit record sets
    record_set_ids = []
    # Fallback to show user how to find available sources
    print("RecordSet list is empty. Consult dataset.metadata.get('distribution', []) for file objects.")

dataframes = {}
if record_set_ids:
    for record_set_id in record_set_ids:
        print(f"Loading data from RecordSet '@id': {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Columns for {record_set_id}: {df.columns.tolist()}")
            display(df.head())  # Show a sample
        else:
            print(f"No records found for record set {record_set_id}.")
    # Pick the first record set for EDA
    focus_record_set_id = record_set_ids[0]
else:
    print('No record sets to load.')

## 4. Exploratory Data Analysis (EDA)
We now apply some basic data processing:
- Filtering records on a numeric field using its `@id` (or column label if field not found)
- Normalizing the numeric column
- Grouping by a categorical field

> **Please update the code below to select the appropriate numeric/categorical field `@id` according to Section 2's output.**

In [ ]:
# --- BEGIN USER INPUT SECTION ---
# Set these to the desired field ids for EDA:
# E.g., numeric_field_id = 'cr:abundance' or column name from the DataFrame

numeric_field_id = None  # e.g. 'cr:abundance' or 'Normalized Abundance'
group_field_id = None    # e.g. 'cr:protein_group' or 'Group'

# List numeric columns and suggest one if the user has not set it
df = None
if dataframes:
    df = dataframes.get(focus_record_set_id)

if df is not None:
    print("DataFrame columns (use one as field @id or column name):")
    print(df.columns.tolist())
    # Suggest a numeric field if not set
    candidate_numeric = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    if not numeric_field_id:
        if candidate_numeric:
            numeric_field_id = candidate_numeric[0]
            print(f"Using {numeric_field_id} as the numeric field.")
        else:
            print("No obvious numeric field found.")
    # Suggest a group field
    candidate_group = [c for c in df.columns if pd.api.types.is_object_dtype(df[c])]
    if not group_field_id:
        if candidate_group:
            group_field_id = candidate_group[0]
            print(f"Using {group_field_id} as the group field.")
        else:
            print("No obvious group/categorical field found.")
    # --- Main EDA Logic ---
    if numeric_field_id and numeric_field_id in df.columns:
        threshold = df[numeric_field_id].quantile(0.75) if not df[numeric_field_id].empty else 0
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered rows where {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())
        # Normalize
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized values for {numeric_field_id}:")
        display(filtered_df[[numeric_field_id, norm_col]].head())
        # Group by
        if group_field_id in filtered_df.columns:
            grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Mean {numeric_field_id} grouped by {group_field_id}:")
            display(grouped.head())
        else:
            print(f"Group field {group_field_id} not found in columns.")
    else:
        print(f"Numeric field {numeric_field_id} not found in DataFrame columns.")
else:
    print("No DataFrame loaded for EDA.")

## 5. Visualization
Let's visualize the distribution of the numeric feature, and explore its relationship with a categorical/grouping field, if selected.

> **You may further edit the code to customize plots as needed.**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id and group_field_id in df.columns:
        # Violin/box plot by group
        top_groups = df[group_field_id].value_counts().index[:5]
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df[df[group_field_id].isin(top_groups)])
        plt.title(f"{numeric_field_id} by {group_field_id} (Top 5 groups)")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No DataFrame loaded or numeric field missing for visualization.")

## 6. Conclusion
In this notebook, you've learned how to load a FAIR⁲ Croissant dataset in Python, review its structure, extract records from record sets by `@id`, and conduct basic EDA and visualization—all referencing attributes by `@id` in accordance with Croissant best practices. 

- We explored available record sets and fields using the metadata.
- We loaded data using the `mlcroissant` library.
- We analyzed and visualized fields using their Croissant schema `@id`s for clarity and reproducibility.

You can now extend these analyses to domain-specific questions relevant to protein abundance and modification studies.

For more details and advanced usage, see the [`mlcroissant` documentation](https://mlcommons.github.io/croissant/python/).